# 501: Quantile-Trajectory-Based Criteria Table

Derives the three no-further-warming criteria from pointwise-in-time quantile trajectories.

Source data: `individual_runs_quantiles_anthropogenic.csv` - gcages'
`CMIP7ScenarioMIPPostProcessor.timeseries_quantile` output (its own native percentile
set: p05/p10/1-6th/p33/p50/p67/5-6ths/p90/p95), extracted directly from the
already-calibrated `individual_runs/` files (see `extract_quantiles_from_calibrated.py`).

For each quantile trajectory, per scenario:

    quantile_peak_year  = year at which that trajectory is maximal
    quantile_at_netzero_co2 = trajectory's value at the scenario's net-zero-CO2 year
    quantile_at_2100       = trajectory's value at 2100
    delta_netzero_co2      = quantile_at_2100 - quantile_at_netzero_co2

    returns_to_or_below_nzco2      = delta_netzero_co2 <= 0
    no_further_warming_after_nzco2 = (quantile_peak_year <= year_netzero_co2) and (delta_netzero_co2 <= 0)
    peak_and_decline_after_nzco2   = (quantile_peak_year >  year_netzero_co2) and (delta_netzero_co2 <  0)

Then, for each scenario_group and each quantile level, the share of scenarios
whose quantile trajectory (at that level) satisfies each criterion - reported the same
way as 501's other percentile-ladder sections (Median = the p50 level's share, with
(p34 to p67)/(p17 to p83)/(p5 to p95) as nested ranges across the other quantile levels.

Outputs:
- `501_Table_1.png`
- `501_STable_1.png`

In [ ]:
import os
from pathlib import Path

import dotenv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

dotenv.load_dotenv()


In [ ]:
# Configuration
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])
PLOT_FOLDER = OUTPUT_FOLDER / "plots"
PLOT_FOLDER.mkdir(parents=True, exist_ok=True)

MANIFEST_ID = "regenerated"
QUANTILE_FILE = OUTPUT_FOLDER / "individual_runs_quantiles_anthropogenic.csv"
NETZERO_TIMINGS_FILE = OUTPUT_FOLDER / "101_netzero_timings.csv"


## Load Quantile Trajectories


In [ ]:
# Load the quantile trajectories - one row per (model, scenario, quantile), year
# columns hold that quantile level's cross-sectional GSAT value at each year.
quantile_df = pd.read_csv(QUANTILE_FILE)
year_cols = [c for c in quantile_df.columns if c.isdigit()]
print(f"Loaded {len(quantile_df)} rows ({quantile_df[['model','scenario']].drop_duplicates().shape[0]} scenarios x {quantile_df['quantile'].nunique()} quantile levels)")
quantile_df.head(3)


## Classify Scenario Groups


In [ ]:
# Classify scenario groups at the (model, scenario, quantile) row level (the
# scenario_group only depends on the scenario name, not the quantile level, so this
# is safe to apply row-wise regardless of granularity).
quantile_df["scenario_group"] = quantile_df["scenario"].apply(
    lambda x: "Stylised Variants (NZCO2-hold)" if "NZCO2" in x
    else ("Stylised Variants (NZGHG-hold)" if "NZKyoto" in x else "Scenarios (NZCO2)")
)

# Build the "Scenarios (NZGHG)" relabeled subset: base scenarios whose (model,
# scenario) also has an NZKyoto-suffixed counterpart - duplicates all 9 quantile rows
# per qualifying scenario, relabeled.
nzkyoto_combos = set(
    quantile_df[quantile_df["scenario_group"] == "Stylised Variants (NZGHG-hold)"]
    .assign(scenario_key=lambda d: d["scenario"].str.replace("_NZKyoto", "", regex=False))
    [["model", "scenario_key"]]
    .itertuples(index=False, name=None)
)
base_df = quantile_df[quantile_df["scenario_group"] == "Scenarios (NZCO2)"]
mask = base_df.apply(lambda r: (r["model"], r["scenario"]) in nzkyoto_combos, axis=1)
base_nzghg = base_df[mask].copy()
base_nzghg["scenario_group"] = "Scenarios (NZGHG)"

quantile_df_extended = pd.concat([quantile_df, base_nzghg], ignore_index=True)

group_counts = (
    quantile_df_extended
    .drop_duplicates(subset=["model", "scenario", "scenario_group"])
    .groupby("scenario_group")
    .size()
)
print("Scenario group summary:")
group_counts


## Merge Net-Zero-CO2 Timing


In [ ]:
# Merge in year_netzero_co2 - looked up by the BASE (unsuffixed) scenario name, since
# 101_netzero_timings.csv only has one row per base (model, scenario), same lookup
# pattern as 500c_calculate_ody.py / extract_quantiles_from_calibrated.py's manifest
# cross-referencing.
netzero_timings = pd.read_csv(NETZERO_TIMINGS_FILE)
netzero_lookup = netzero_timings.set_index(["model", "scenario"])["year_netzero_co2"].to_dict()

quantile_df_extended["base_scenario"] = (
    quantile_df_extended["scenario"].str.replace("_NZCO2", "", regex=False).str.replace("_NZKyoto", "", regex=False)
)
quantile_df_extended["year_netzero_co2"] = quantile_df_extended.apply(
    lambda r: netzero_lookup.get((r["model"], r["base_scenario"])), axis=1
)

n_missing = quantile_df_extended["year_netzero_co2"].isna().sum()
print(f"Rows with year_netzero_co2: {len(quantile_df_extended) - n_missing} / {len(quantile_df_extended)} ({n_missing} missing)")

quantile_df_extended = quantile_df_extended[quantile_df_extended["year_netzero_co2"].notna()].copy()
quantile_df_extended["year_netzero_co2"] = quantile_df_extended["year_netzero_co2"].astype(int)


## Derive Criteria from Quantile Trajectories


In [ ]:
# Derive the three criteria directly from each quantile trajectory's own peak-year and
# NZCO2/2100 endpoint values.
year_values = quantile_df_extended[year_cols].values
quantile_df_extended["quantile_peak_year"] = quantile_df_extended[year_cols].idxmax(axis=1).astype(int)
quantile_df_extended["quantile_peak_value"] = quantile_df_extended[year_cols].max(axis=1)

# Row-wise lookup of the trajectory's value at that scenario's own year_netzero_co2
# (varies per row) - vectorized via integer position indexing rather than .apply().
year_col_positions = {int(c): i for i, c in enumerate(year_cols)}
row_positions = quantile_df_extended["year_netzero_co2"].map(year_col_positions).values
quantile_df_extended["quantile_at_netzero_co2"] = year_values[np.arange(len(year_values)), row_positions]
quantile_df_extended["quantile_at_2100"] = quantile_df_extended["2100"]

quantile_df_extended["delta_netzero_co2"] = (
    quantile_df_extended["quantile_at_2100"] - quantile_df_extended["quantile_at_netzero_co2"]
)

peak_before_nzco2 = quantile_df_extended["quantile_peak_year"] <= quantile_df_extended["year_netzero_co2"]
quantile_df_extended["no_further_warming_after_nzco2"] = peak_before_nzco2 & (quantile_df_extended["delta_netzero_co2"] <= 0)
quantile_df_extended["peak_and_decline_after_nzco2"] = (~peak_before_nzco2) & (quantile_df_extended["delta_netzero_co2"] <= 0)
# "Peak-and-decline" means decline after peak to OR BELOW the NZCO2-level, so this
# uses <=0 (not strict <0) - consistent with no_further_warming_after_nzco2's own
# threshold above. With both sub-criteria using <=0, their union is now provably
# identical to delta<=0 in every case (no edge case at all, not just empirically -
# still expressed as a union here for clarity/robustness, matching 501's convention).
quantile_df_extended["returns_to_or_below_nzco2"] = (
    quantile_df_extended["no_further_warming_after_nzco2"] | quantile_df_extended["peak_and_decline_after_nzco2"]
)

quantile_df_extended[[
    "model", "scenario", "scenario_group", "quantile", "year_netzero_co2",
    "quantile_peak_year", "delta_netzero_co2", "returns_to_or_below_nzco2",
    "no_further_warming_after_nzco2", "peak_and_decline_after_nzco2",
]].head(9)


## Aggregate: Share of Scenarios per Quantile Level


In [ ]:
# Map gcages' exact quantile values to display labels (rounded for readability - the
# underlying quantile values used are still the exact 1/6, 5/6 etc., only the label
# text is rounded to p17/p83).
QUANTILE_LABELS = {
    0.05: "p05", 0.10: "p10", 1.0 / 6.0: "p17", 0.33: "p33", 0.5: "p50",
    0.67: "p67", 5.0 / 6.0: "p83", 0.90: "p90", 0.95: "p95",
}
# Map to nearest key (floating point quantile column from CSV round-trip)
def _nearest_label(q):
    nearest = min(QUANTILE_LABELS, key=lambda k: abs(k - q))
    return QUANTILE_LABELS[nearest]

quantile_df_extended["quantile_label"] = quantile_df_extended["quantile"].apply(_nearest_label)

CRITERIA_COLS = {
    "returns to NZCO2 or below": "returns_to_or_below_nzco2",
    "no further warming": "no_further_warming_after_nzco2",
    "peak-and-decline": "peak_and_decline_after_nzco2",
}

groups = ["Scenarios (NZCO2)", "Scenarios (NZGHG)", "Stylised Variants (NZCO2-hold)", "Stylised Variants (NZGHG-hold)"]

# Pooled share/probability - % of scenarios (in the group) whose quantile trajectory
# satisfies the criterion, at each quantile level.
quantile_trajectory_shares_mean = {}

for group in groups:
    gdf = quantile_df_extended[quantile_df_extended["scenario_group"] == group]
    rows_mean = {}
    for label in QUANTILE_LABELS.values():
        sub = gdf[gdf["quantile_label"] == label]
        rows_mean[label] = {crit_label: sub[crit_col].mean() * 100 for crit_label, crit_col in CRITERIA_COLS.items()}
    quantile_trajectory_shares_mean[group] = pd.DataFrame(rows_mean).T

quantile_trajectory_shares_mean["Scenarios (NZCO2)"]


## Aggregate: Post-NZCO2 GSAT Change per Quantile Level


In [ ]:
# Post-NZCO2 GSAT change by 2100, using the same quantile-trajectory logic as the
# criteria above - median across scenarios (not mean, consistent with 501's own
# pooled_delta_stats, which uses median+percentiles for this continuous, skewed
# quantity) of delta_netzero_co2, at each fixed quantile level.
gsat_change_by_group = {}
for group in groups:
    gdf = quantile_df_extended[quantile_df_extended["scenario_group"] == group]
    rows = {}
    for label in QUANTILE_LABELS.values():
        sub = gdf[gdf["quantile_label"] == label]
        rows[label] = sub["delta_netzero_co2"].median()
    gsat_change_by_group[group] = pd.Series(rows)

pd.DataFrame(gsat_change_by_group)


## Aggregate: Post-NZCO2 Overshoot Degree-Years per Quantile Level


In [ ]:
# Post-NZCO2 overshoot degree-years, using the same quantile-trajectory logic as
# above: for each scenario's quantile trajectory, integrate the excess above that
# trajectory's own value at year_netzero_co2, from year_netzero_co2 to 2100 (same
# definition as 500c_calculate_ody.py, applied to a quantile trajectory instead of a
# real member's timeseries) - then median across scenarios at each fixed quantile
# level, consistent with the GSAT-change aggregation above.
years_arr = np.array([int(c) for c in year_cols])
year_nz_arr = quantile_df_extended["year_netzero_co2"].values
year_values_full = quantile_df_extended[year_cols].values

window_mask = (years_arr[None, :] >= year_nz_arr[:, None]) & (years_arr[None, :] <= 2100)
baseline_arr = quantile_df_extended["quantile_at_netzero_co2"].values[:, None]
excess = np.clip(year_values_full - baseline_arr, a_min=0, a_max=None)
quantile_df_extended["ody_quantile"] = np.where(window_mask, excess, 0).sum(axis=1)

ody_by_group = {}
for group in groups:
    gdf = quantile_df_extended[quantile_df_extended["scenario_group"] == group]
    rows = {}
    for label in QUANTILE_LABELS.values():
        sub = gdf[gdf["quantile_label"] == label]
        rows[label] = sub["ody_quantile"].median()
    ody_by_group[group] = pd.Series(rows)

pd.DataFrame(ody_by_group)


## Load Per-Member Data for ZEC50 Substitution


In [ ]:
# Load per-member component metrics for the ZEC50 substitution test below - this needs
# real member-level data (each member's own paired total and CO2_NZCO2 values), not
# the quantile-trajectory file, since the substitution has to happen at the member
# level first (see markdown above) to avoid combining two independently-extracted
# quantile trajectories, which would not be mathematically valid (percentile of a sum
# is not the sum of percentiles in general).
METRICS_FILE = OUTPUT_FOLDER / f"500b_component_metrics_extended_{MANIFEST_ID}.csv"
member_df = pd.read_csv(
    METRICS_FILE,
    usecols=["model", "scenario", "run_id", "scenario_group", "year_netzero_co2",
             "GSAT|ANTHROPOGENIC|delta_netzero_co2", "GSAT|CO2_NZCO2|delta_netzero_co2"],
)
print(f"Loaded {len(member_df):,} member rows")


## ZEC50 Substitution at Member Level


In [ ]:
# ZEC50 substitution at the member level: for each real member, subtract its own
# diagnosed CO2_NZCO2 contribution and add back a fixed assessed value (rescaled from
# IPCC AR6's standardized 50-year window to this member's own actual window length),
# matching 501's own substitution formula exactly. Caveat: this applies the SAME
# assessed value to every member of a scenario, so it cannot represent genuine
# member-to-member variation in a counterfactual ZEC - it only answers "what if the
# model's own diagnosed ZEC* were replaced by this one fixed assessed number," for
# every member alike.
ZEC50_VALUES = {"low": -0.19, "central": 0.0, "high": 0.19}

for label, zec50_val in ZEC50_VALUES.items():
    assessed_rescaled = zec50_val * (2100 - member_df["year_netzero_co2"]) / 50
    substituted_delta = (
        member_df["GSAT|ANTHROPOGENIC|delta_netzero_co2"]
        - member_df["GSAT|CO2_NZCO2|delta_netzero_co2"]
        + assessed_rescaled
    )
    member_df[f"substituted_delta_{label}"] = substituted_delta

member_df[["model", "scenario", "run_id", "substituted_delta_low", "substituted_delta_central", "substituted_delta_high"]].head(5)


## Aggregate: ZEC50-Substituted Share per Quantile Level


In [ ]:
# Aggregate: for each scenario, at each quantile level, is the q-th-percentile
# member's substituted delta <=0 - then share of scenarios (mean) at that fixed q,
# for each of the three ZEC50 assessed values. Same aggregation logic as criterion (I)
# above, applied to the substituted quantity instead of the original.
zec_sensitivity_by_group = {}
for group in groups:
    gdf = member_df[member_df["scenario_group"] == group]
    rows = {}
    for q, label in QUANTILE_LABELS.items():
        row = {}
        for zec_label in ZEC50_VALUES:
            col = f"substituted_delta_{zec_label}"
            per_scenario_q = gdf.groupby(["model", "scenario"])[col].quantile(q)
            row[zec_label] = (per_scenario_q <= 0).mean() * 100
        rows[label] = row
    zec_sensitivity_by_group[group] = pd.DataFrame(rows).T

zec_sensitivity_by_group["Scenarios (NZCO2)"]


## Quantile-Trajectory-Based Criteria Table


In [ ]:
rename_map = {
    "Scenarios (NZCO2)":              "Scenarios\n(NZCO2)",
    "Scenarios (NZGHG)":              "Scenarios\n(NZGHG)",
    "Stylised Variants (NZCO2-hold)": "Stylised Variants\n(NZCO2-hold)",
    "Stylised Variants (NZGHG-hold)": "Stylised Variants\n(NZGHG-hold)",
}


def fmt_pct(x):
    return f"{x:.1f}"


def fmt_degc(x):
    return f"{x:.2f}"


def fmt_ody(x):
    return f"{x:.2f}"


def pctl_rows(stats_by_group, key):
    """Build the standard Median/(p33-p67)/(p17-p83)/(p5-p95) row block for a
    {group: DataFrame-or-Series indexed by quantile label} stats dict, selecting
    column `key` when the per-group object is a DataFrame (criteria/ZEC shares),
    or using the Series directly when `key` is None (GSAT change/ODY)."""
    def val(g, label):
        obj = stats_by_group[g]
        return obj.loc[label, key] if key is not None else obj[label]

    fmt = fmt_pct if key is not None else (fmt_degc if stats_by_group is gsat_change_by_group else fmt_ody)
    return [
        ("Median", [fmt(val(g, "p50")) for g in groups]),
        ("(p33 to p67)", [f"({fmt(val(g, 'p33'))} to {fmt(val(g, 'p67'))})" for g in groups]),
        ("(p17 to p83)", [f"({fmt(val(g, 'p17'))} to {fmt(val(g, 'p83'))})" for g in groups]),
        ("(p5 to p95)", [f"({fmt(val(g, 'p05'))} to {fmt(val(g, 'p95'))})" for g in groups]),
    ]


# All available table sections, keyed by a stable (unlettered) identifier - letters
# are assigned fresh, sequentially, by build_table() below, so each table (main or
# supplementary) gets its own self-contained a), b), c)... numbering.
ALL_SECTIONS = {
    "returns_to_nzco2": (
        "Share of scenarios with\nGSAT(t=2100) at or below\nGSAT(t=NZCO2) [%]",
        pctl_rows(quantile_trajectory_shares_mean, "returns to NZCO2 or below"),
    ),
    "no_further_warming": (
        "Share of scenarios with peak\nGSAT before or at NZCO2 and \nGSAT(t=2100)≤GSAT(t=NZCO2) [%]",
        pctl_rows(quantile_trajectory_shares_mean, "no further warming"),
    ),
    "peak_and_decline": (
        "Share of scenarios with peak\nGSAT after NZCO2 and \nGSAT(t=2100)≤GSAT(t=NZCO2) [%]",
        pctl_rows(quantile_trajectory_shares_mean, "peak-and-decline"),
    ),
    "gsat_change": (
        "Post-NZCO2 GSAT\nchange by 2100 [°C]",
        pctl_rows(gsat_change_by_group, None),
    ),
    "ody": (
        "Post-NZCO2 overshoot\ndegree-years [°C·years]",
        pctl_rows(ody_by_group, None),
    ),
    "zec_low": (
        "As a), having substituted\nZEC$_{50}$ with the IPCC AR6\nassessed low value (-0.19°C) [%]",
        pctl_rows(zec_sensitivity_by_group, "low"),
    ),
    "zec_central": (
        "As a), having substituted\nZEC$_{50}$ with the IPCC AR6\nassessed central value (0°C) [%]",
        pctl_rows(zec_sensitivity_by_group, "central"),
    ),
    "zec_high": (
        "As a), having substituted\nZEC$_{50}$ with the IPCC AR6\nassessed high value (0.19°C) [%]",
        pctl_rows(zec_sensitivity_by_group, "high"),
    ),
}


def build_table(section_keys, output_filename, col_header_height = 0.08, section_header_height = 0.14):
    """Build and save one criteria table containing only the given section keys (in
    order), each freshly lettered a), b), c)... within this table."""
    import string
    sections_q = [
        (f"{letter}) {ALL_SECTIONS[key][0]}", ALL_SECTIONS[key][1])
        for letter, key in zip(string.ascii_lowercase, section_keys)
    ]

    col_labels_q = [rename_map[g] for g in groups]
    n_cols_q = len(col_labels_q)

    row_labels_q = []
    cell_rows_q = []
    section_row_indices_q = []
    current_table_row = 1  # table row 0 = column headers
    for label, rows in sections_q:
        section_row_indices_q.append(current_table_row)
        row_labels_q.append(label)
        cell_rows_q.append([""] * n_cols_q)
        current_table_row += 1
        for r_label, r_vals in rows:
            row_labels_q.append(r_label)
            cell_rows_q.append(r_vals)
            current_table_row += 1

    n_rows_q = len(row_labels_q)

    fig, ax = plt.subplots(figsize=(8, n_rows_q * 0.3 + 0.5))
    ax.axis("off")

    table = ax.table(
        cellText=cell_rows_q,
        colLabels=col_labels_q,
        rowLabels=row_labels_q,
        loc="center",
        cellLoc="center",
        colLoc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.8)

    # Column header row is 2 lines ("Scenarios\n(NZCO2)" etc.); section header rows
    # are 3 lines - each needs enough height for its own line count to avoid
    # overlapping the row below.
    col_header_height = col_header_height
    for col_idx in range(n_cols_q):
        table[0, col_idx].set_height(col_header_height)
        table[0, col_idx].get_text().set_fontweight("bold")
        table[0, col_idx].set_facecolor("#e0e0e0")

    section_header_height = section_header_height
    for row_idx in section_row_indices_q:
        for col_idx in range(n_cols_q):
            table[row_idx, col_idx].set_height(section_header_height)
            table[row_idx, col_idx].set_facecolor("#c0c0c0")
            table[row_idx, col_idx].get_text().set_fontweight("bold")
        table[row_idx, -1].set_height(section_header_height)
        table[row_idx, -1].get_text().set_fontweight("bold")
        table[row_idx, -1].set_facecolor("#c0c0c0")
        table[row_idx, -1].get_text().set_ha("left")

    plt.savefig(PLOT_FOLDER / output_filename, dpi=300, bbox_inches="tight")
    plt.show()


# Main table: drops what were sections b), c), e) (no-further-warming,
# peak-and-decline, overshoot-degree-years) - those move to the supplementary table.
build_table(
    ["returns_to_nzco2", "gsat_change", "zec_low", "zec_central", "zec_high"],
    "501_Table_1.png",
    col_header_height = 0.07,
    section_header_height = 0.09
)

# Supplementary table: only the three sections dropped from the main table above.
build_table(
    ["no_further_warming", "peak_and_decline", "ody"],
    "501_STable_1.png",
    col_header_height = 0.1,
    section_header_height = 0.14
)